# Selection XGB

# Setup

## Imports


In [ ]:
import logging
import pathlib
import warnings

import matplotlib.pyplot as plt
import pandas as pd
import yaml

from thesis_project import config as global_config
from thesis_project.importance_xgb import data_paths, evaluation, settings

## Paths

In [ ]:
TICKER = "fbtp"
# NOTE: Use "notebook-name" if needed
NOTEBOOK = None

# Lobs directories
LOB_DIR = global_config.RAW_DIR / TICKER
LOB_FUT_DIR = LOB_DIR / "futures"
LOB_CTD_DIR = LOB_DIR / "ctd"

# Output directories
if NOTEBOOK is not None:
    OUT_FIG_DIR = global_config.FIG_DIR / TICKER / NOTEBOOK
    OUT_FIG_DIR.mkdir(parents=True, exist_ok=True)
    OUT_TAB_DIR = global_config.TAB_DIR / TICKER / NOTEBOOK
    OUT_TAB_DIR.mkdir(parents=True, exist_ok=True)
    # NOTE: Create a purpose specific directory if needed
    # OUT_INT_DIR = config.INT_DIR / TICKER / NOTEBOOK
    # OUT_INT_DIR.mkdir(parents=True, exist_ok=True)
    # OUT_PRO_DIR = config.PRO_DIR / TICKER / NOTEBOOK
    # OUT_PRO_DIR.mkdir(parents=True, exist_ok=True)

## Settings

In [ ]:
# Matplotlib global settings
# config.MPL contains a personalized default configuration for matplotlib
plt.rcdefaults()
plt.style.use("seaborn-v0_8-paper")

In [ ]:
# Warnings settings
warnings.simplefilter("always", UserWarning)

In [ ]:
logger = logging.getLogger(__name__)

## Utilities

# Main

## Common

## Load Data

In [ ]:
run = "004"
dir = f"run_{run}"
base_path = global_config.INT_DIR / TICKER / "importance-xgb" / "runs"
path = base_path / dir

with open(path / "config.yaml") as f:
    config_dict = yaml.safe_load(f)
exp_config_obj = settings.AnalysisConfig(**config_dict)

with open(path / "data_paths.yaml") as f:
    data_paths_dict = yaml.safe_load(f)
    target = next((name, pathlib.Path(path)) for name, path in data_paths_dict["target"].items())
exp_data_paths = data_paths.DataPathsConfig(
    ticker=TICKER, target=target, features=data_paths_dict["features"]
)

with open(path / "metadata.yaml") as f:
    exp_metadata = yaml.safe_load(f)

selection_hist_dict = {}
for seed in exp_metadata["seeds"]:
    selection_hist_dict[seed] = pd.read_csv(path / f"selection_history_seed_{seed}.csv")

ranking_hist_dict = {}
for seed in exp_metadata["seeds"]:
    ranking_hist_dict[seed] = pd.read_excel(path / f"ranking_history_seed_{seed}.xlsx")

## Metrics

In [ ]:
metric = "rmse"
evaluation.selection_xgb_metrics(selection_hist_dict, metric)

In [ ]:
metric = "r2"
evaluation.selection_xgb_metrics(selection_hist_dict, metric)

## XGBoost Importance

In [ ]:
n = 25
metric = "gain"
evaluation.selection_xgb_mean_ranking(selection_hist_dict, n, metric)